# 02 — Preprocess LEMON fMRI & Extract Features

For each subject this notebook:
1. Downloads the raw 4D BOLD from OpenNeuro S3
2. Computes a **mean BOLD volume** (average across timepoints)
3. Computes an **fALFF map** (fractional Amplitude of Low-Frequency Fluctuations) — the primary resting-state biomarker
4. Resamples both to a standard 48×48×32 MNI volume
5. Saves a `(2, 48, 48, 32)` NumPy array per subject — 2 channels: mean BOLD + fALFF
6. Deletes the raw file to free disk space

**Output:** `lemon/features/sub-XXXXXX.npy` per subject (~1.4 MB each, ~300 MB total)

In [ ]:
!pip install -q nibabel nilearn numpy scipy tqdm awscli

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
BASE_DIR    = '/content/drive/MyDrive/lemon'
RAW_DIR     = f'{BASE_DIR}/raw'
FEAT_DIR    = f'{BASE_DIR}/features'
os.makedirs(RAW_DIR,  exist_ok=True)
os.makedirs(FEAT_DIR, exist_ok=True)

with open(f'{BASE_DIR}/subject_list.json') as f:
    subjects = json.load(f)

BUCKET = 's3://openneuro.org/ds000221'
print(f'Subjects to process: {len(subjects)}')

In [ ]:
import numpy as np
from scipy import signal

# ── fALFF implementation ──────────────────────────────────────────────────────
# fALFF = power in low-frequency band (0.01–0.08 Hz) / total power
# This captures spontaneous BOLD fluctuations tied to resting-state networks.

def compute_falff(bold_data, tr, low=0.01, high=0.08):
    """
    bold_data : (x, y, z, t) float32 array
    tr        : repetition time in seconds
    Returns   : (x, y, z) fALFF map
    """
    n_t   = bold_data.shape[3]
    freqs = np.fft.rfftfreq(n_t, d=tr)  # frequency axis

    # FFT along time axis
    fft_data = np.abs(np.fft.rfft(bold_data, axis=3))

    low_mask  = (freqs >= low)  & (freqs <= high)
    total_pow = fft_data.sum(axis=3) + 1e-8
    low_pow   = fft_data[..., low_mask].sum(axis=3)

    return (low_pow / total_pow).astype(np.float32)

print('fALFF function defined ✓')

In [ ]:
import nibabel as nib
from nilearn import image as nli
from nilearn.maskers import NiftiMasker

# Standard MNI target space — small enough for CNN, large enough to preserve anatomy
TARGET_SHAPE  = (48, 48, 32)
TARGET_AFFINE = np.diag([4.0, 4.0, 4.5, 1.0])  # 4×4×4.5 mm voxels

def preprocess_subject(bold_path):
    """
    Load 4D BOLD, compute mean + fALFF, resample to target space.
    Returns (2, 48, 48, 32) float32 array: channel 0 = mean BOLD, channel 1 = fALFF.
    """
    img = nib.load(bold_path)
    tr  = float(img.header.get_zooms()[3])  # repetition time
    if tr == 0:
        tr = 1.4  # LEMON TR is 1.4s if header is missing

    data = img.get_fdata(dtype=np.float32)  # (x, y, z, t)

    # ── Mean BOLD ────────────────────────────────────────────────────────────
    mean_vol = data.mean(axis=3)
    mean_img = nib.Nifti1Image(mean_vol, img.affine)

    # ── fALFF ────────────────────────────────────────────────────────────────
    # Detrend first to remove scanner drift
    from scipy.signal import detrend
    data_dt = detrend(data, axis=3, type='linear')
    falff   = compute_falff(data_dt, tr)
    falff_img = nib.Nifti1Image(falff, img.affine)

    # ── Resample both to target MNI space ────────────────────────────────────
    mean_res  = nli.resample_img(mean_img,  target_affine=TARGET_AFFINE, target_shape=TARGET_SHAPE)
    falff_res = nli.resample_img(falff_img, target_affine=TARGET_AFFINE, target_shape=TARGET_SHAPE)

    # ── Normalise each channel to zero mean, unit std ─────────────────────────
    def norm(arr):
        arr = arr.astype(np.float32)
        m, s = arr.mean(), arr.std()
        return (arr - m) / (s + 1e-8)

    ch0 = norm(mean_res.get_fdata(dtype=np.float32))
    ch1 = norm(falff_res.get_fdata(dtype=np.float32))

    # Stack: (2, 48, 48, 32)
    return np.stack([ch0, ch1], axis=0)

print('Preprocessing function defined ✓')

In [ ]:
import subprocess, traceback
from tqdm import tqdm

failed = []
skipped = 0

for sub in tqdm(subjects, desc='Subjects'):
    out_path = f'{FEAT_DIR}/{sub}.npy'

    # Skip already processed subjects (allows resuming after Colab disconnect)
    if os.path.exists(out_path):
        skipped += 1
        continue

    raw_path = f'{RAW_DIR}/{sub}_bold.nii.gz'

    try:
        # 1. Download
        bold_key = f'{BUCKET}/{sub}/func/{sub}_task-rest_bold.nii.gz'
        result = subprocess.run(
            ['aws', 's3', 'cp', bold_key, raw_path, '--no-sign-request'],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(f'S3 download failed: {result.stderr}')

        # 2. Preprocess
        features = preprocess_subject(raw_path)

        # 3. Save features
        np.save(out_path, features)

    except Exception as e:
        print(f'\n  FAILED {sub}: {e}')
        failed.append(sub)

    finally:
        # 4. Always delete raw file to free disk space
        if os.path.exists(raw_path):
            os.remove(raw_path)

print(f'\nDone. Processed: {len(subjects)-len(failed)-skipped}  Skipped: {skipped}  Failed: {len(failed)}')
if failed:
    print('Failed subjects:', failed)
    with open(f'{BASE_DIR}/failed_subjects.json', 'w') as f:
        json.dump(failed, f)

In [ ]:
# ── Verify output ─────────────────────────────────────────────────────────────
feat_files = [f for f in os.listdir(FEAT_DIR) if f.endswith('.npy')]
print(f'Feature files saved: {len(feat_files)}')

# Check one
sample = np.load(f'{FEAT_DIR}/{feat_files[0]}')
print(f'Shape: {sample.shape}  — expected (2, 48, 48, 32)')
print(f'Channel 0 (mean BOLD): min={sample[0].min():.3f}  max={sample[0].max():.3f}')
print(f'Channel 1 (fALFF):     min={sample[1].min():.3f}  max={sample[1].max():.3f}')

# Quick visual check — show middle slice of mean BOLD
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(sample[0, :, :, 16].T, cmap='gray', origin='lower')
axes[0].set_title('Mean BOLD — axial slice z=16')
axes[1].imshow(sample[1, :, :, 16].T, cmap='hot', origin='lower')
axes[1].set_title('fALFF — axial slice z=16')
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/sample_features.png', dpi=100)
plt.show()
print('\nNext step: open 03_build_dataset.ipynb')